### 1. **Необходимость, достаточность, причинность**

Вчера на лекции были описаны  пять вопросов к любой защите. Наша задача сегодня сфокусироваться на третьем: **гарантия или сигнал?** в чуть более общем смысле:

— **Насколько возможно** получить **гарантию** того, что $x$ появило на результат $y$
- **Как отличить гарантию** от сигнала
- **Как получить** наиболее значимый сигнал


Пойдем от вчершанего примера с почтой. Пусть у нас есть фильтр входящих писем. Эмпирически мы выяснили, что фильтр ловит 99 %, его пробуют пробить 100 раз, и хотя бы одна попытка проходит с вероятностью 63% ($1-0,99^{100}$). 

Перейдем на этой задаче в контекст интерпретируемости: пусть фильтр тут играет не роль щита, а роль способа понять «что повлияло на ответ/действие/состояние»?

Отсюда первое:
— Фильтр для защиты — это одно
— Способ проанилизровать — "что повлияло" — это другое. 

Для ответа на вопрос «что повлияло» нам нужны две компоненты:
- **необходимость**: не было бы $x$ $\Rightarrow$ не было бы и $y$ ($\neg x \Rightarrow \neg y$);
- **достаточность**: был бы $x$ $\Rightarrow$ был бы и $y$ ($x \Rightarrow y$).


Строгие постановки этой идеи:
- J. Pearl, *Probabilities of causation: three counterfactual interpretations
  and their identification*, Synthese 121, 1999.
  [r260-reprint.pdf](https://ftp.cs.ucla.edu/pub/stat_ser/r260-reprint.pdf)
- D. Watson, L. Gultchin, A. Taly, L. Floridi, *Local explanations via necessity
  and sufficiency: unifying theory and practice*, UAI 2021.
  [PMLR v161](https://proceedings.mlr.press/v161/watson21a.html)

## Вероятностный смысл
(Pearl 1999, опр. 7–9)

Пусть $x$ — причина (инструмент был, правило сработало), $y$ — исход (ответ получен, атака заблокирована), $Y_{x'}$ — каким был бы исход, если бы $x$ не было.

**Вероятность необходимости.** $x$ был, $y$ случился. Какова вероятность, что без $x$ исхода бы не было:

$$\text{PN} = P(Y_{x'} = y' \mid X = x,\ Y = y)$$

**Вероятность достаточности.** $x$ не было, $y$ не случился. Какова вероятность, что с $x$ исход появился бы:

$$\text{PS} = P(Y_{x} = y \mid X = x',\ Y = y')$$

**Необходимость и достаточность сразу:** $\text{PNS} = P(Y_x = y,\ Y_{x'} = y')$.

**Лемма 1:** $\ \text{PNS} = P(x, y)\cdot\text{PN} + P(x', y')\cdot\text{PS}$

<details>
<summary><b>Доказательство:</b> формула полной вероятности + согласованность</summary>

Разбиваем по четырём гипотезам о том, что наблюдали:

$$
\begin{aligned}
\text{PNS} = P(Y_x = y,\ Y_{x'} = y')
&= P(x, y)\cdot P(Y_x = y,\ Y_{x'} = y' \mid x, y) \\
&+ P(x, y')\cdot P(Y_x = y,\ Y_{x'} = y' \mid x, y') \\
&+ P(x', y)\cdot P(Y_x = y,\ Y_{x'} = y' \mid x', y) \\
&+ P(x', y')\cdot P(Y_x = y,\ Y_{x'} = y' \mid x', y') \\[4pt]
&= P(x, y)\cdot \underbrace{P(Y_{x'} = y' \mid x, y)}_{\text{PN}}
 \;+\; P(x, y')\cdot 0
 \;+\; P(x', y)\cdot 0
 \;+\; P(x', y')\cdot \underbrace{P(Y_{x} = y \mid x', y')}_{\text{PS}}
\end{aligned}
$$

Переход во второй строке держится на **аксиоме согласованности**: если
наблюдали $X = x$, то контрфактуал при этом же значении совпадает с фактом,
$Y_x = Y$. Отсюда:

- при $(x, y)$ условие $Y_x = y$ уже выполнено, остаётся
  $P(Y_{x'} = y' \mid x, y)$ — это PN;
- при $(x', y')$ уже выполнено $Y_{x'} = y'$, остаётся
  $P(Y_x = y \mid x', y')$ — это PS;
- при $(x, y')$ согласованность даёт $Y_x = y' \ne y$, событие невозможно — ноль;
- при $(x', y)$ она даёт $Y_{x'} = y \ne y'$, тоже ноль.

PNS считается только по тем случаям, где исход совпал с причиной: в них
необходимость и достаточность — одно и то же событие, увиденное с двух сторон.

</details>


In [1]:
from itertools import combinations, product
from math import factorial

import numpy as np
import pandas as pd

pd.set_option("display.precision", 3)

## Пример: кто остановил EchoLeak

**Зачем этот пример.** Вчера (слайды 10–11, 16–17) EchoLeak можно было остановить
двумя способами: фильтром входящих писем ($F$, ловит 99 % — сигнал) и запретом
грузить адреса из ответа ($B$ — гарантия). Допустим, включены оба, и утечки
не было. Естественный вопрос для отчёта или разбора инцидента: **кто её
остановил?**

На этом вопросе мы упираемся в проблему **избыточности** (в литературе о
причинности — *сверхдетерминации*): исход обеспечен двумя причинами сразу.
Тогда «кто остановил» распадается на два разных вопроса — хватило бы каждой
защиты по отдельности (достаточность) и обошлись бы мы без каждой
(необходимость) — и ответы на них расходятся. А ответ на второй вопрос
в нашем конкретном случае — контрфактуал: то же письмо с выключенной
защитой никто не видел. Посчитать его можно, только зная модель мира.
Здесь мы её знаем, потому что сами её записали.

**Модель.** $X_F, X_B \in \{0, 1\}$ — включена ли защита; заранее мы этого
не знаем, каждая включена с вероятностью $1/2$. $U$ — распознал ли фильтр
именно это письмо, $P(U = 1) = 0.99$. Исход «утечки нет»:

$$Y = (X_F \land U) \lor X_B$$


In [2]:
def exact_shapley(v, players):
    n = len(players)
    return {i: sum(factorial(k) * factorial(n - k - 1) / factorial(n)
                   * (v(frozenset(S) | {i}) - v(frozenset(S)))
                   for k in range(n) for S in combinations([p for p in players if p != i], k))
            for i in players}

P_ON, P_CATCH = 0.5, 0.99
worlds = {(F, B, U): (P_ON if F else 1 - P_ON) * (P_ON if B else 1 - P_ON) * (P_CATCH if U else 1 - P_CATCH)
          for F, B, U in product([0, 1], repeat=3)}

def outcome(world, **do):
    # Y в мире world = (X_F, X_B, U); do фиксирует защиты: контрфактуал «тот же мир, другая защита»
    w = dict(zip("FBU", world)) | do
    return int((w["F"] and w["U"]) or w["B"])

def probabilities_of_causation(c):
    k = "FB".index(c)
    xy = {w: p for w, p in worlds.items() if w[k] == 1 and outcome(w) == 1}      # защита была, утечки нет
    x_y_ = {w: p for w, p in worlds.items() if w[k] == 0 and outcome(w) == 0}    # защиты не было, утечка
    pn = sum(p for w, p in xy.items() if outcome(w, **{c: 0}) == 0) / sum(xy.values())
    ps = sum(p for w, p in x_y_.items() if outcome(w, **{c: 1}) == 1) / sum(x_y_.values())
    pns = sum(p for w, p in worlds.items() if outcome(w, **{c: 1}) == 1 and outcome(w, **{c: 0}) == 0)
    assert abs(pns - (sum(xy.values()) * pn + sum(x_y_.values()) * ps)) < 1e-12   # лемма 1
    return pn, ps, pns

def pn_in_our_world(c):
    # наш случай: обе защиты включены, утечки не было; неизвестно только U
    ours = {w: p for w, p in worlds.items() if w[0] == 1 and w[1] == 1}
    return sum(p for w, p in ours.items() if outcome(w, **{c: 0}) == 0) / sum(ours.values())

def v(S):
    # функция ценности как в SHAP: защиты из S включены, остальное из фона
    return sum(p * outcome(w, **{c: 1 for c in S}) for w, p in worlds.items())

phi = exact_shapley(v, ["F", "B"])
table = pd.DataFrame({c: (*probabilities_of_causation(c), pn_in_our_world(c), phi[c]) for c in "FB"},
                     index=["PN", "PS", "PNS", "PN в нашем случае", "Шепли"]).T
table.index = ["F: фильтр (сигнал)", "B: запрет адресов (гарантия)"]
table


,PN,PS,PNS,PN в нашем случае,Шепли
F: фильтр (сигнал),0.497,0.99,0.495,0.00,0.124
B: запрет адресов (гарантия),0.505,1.00,0.505,0.01,0.129


Что здесь видно:

- **Достаточность** у обеих защит почти единица: каждая по отдельности
  останавливает утечку (фильтр — в 99 % писем).
- **Необходимость в нашем случае**, когда включены обе: у фильтра ноль
  (запрет держит и без него), у запрета 0.01 (без него утечка случилась бы
  только в тех 1 % писем, которые фильтр пропустил). Ни одна защита
  по отдельности не необходима.
- **Шепли** делит вклад почти поровну, 0.124 и 0.129 — так же, как почти
  поровну распределена достаточность. Ответ «обе примерно одинаково
  важны» не ошибка вычисления. Это честный ответ на **другой вопрос**.

Это **теорема об избыточности**: если $i$ и $j$ взаимозаменяемы, симметрия
и эффективность вынуждают $\varphi_i = \varphi_j > 0$, хотя
$\text{PN}(i) = \text{PN}(j) = 0$. При $k$ взаимозаменяемых компонентах
каждый получает $1/k$ от общего вклада: чем больше перекрытие правил
детектора, тем ничтожнее выглядит каждое. На какой вопрос отвечает
Шепли — объясняет Watson et al.


## Что меряет функция ценности SHAP

Watson et al. (2021) определяют для фактора $c$ и исхода $y$ на распределении $D$
(опр. 2 и 3):

$$\text{PS}(c, y) = P_D\big(f(z) = y \mid c(z) = 1\big), \qquad
  \text{PN}(c, y) = P_D\big(c(z) = 1 \mid f(z) = y\big)$$

— precision и recall воображаемого классификатора «$c$ предсказывает $y$».
Это не контрфактуальные PN и PS Пёрла, но ровно та же пара вопросов:
«хватает ли $c$» и «обходится ли $y$ без $c$».

**Proposition 1.** Пусть $c_S(z) = 1$, если $z$ построен фиксацией $x_S$
и сэмплированием остальных признаков из $D(\cdot \mid S)$. Тогда
$v(S) = \text{PS}(c_S, y)$.

То есть значение Шепли — средний прирост **достаточности** случайного
подмножества при добавлении признака. Вопрос «без чего ответ был бы другим»
в этой конструкции не задаётся. 

## Типичный ML-кейс: ирисы

Классификатор цветов по четырём признакам. Обучаем на 70 % выборки,
объясняем цветы из отложенных 30 %; фон $D$ для объяснений — обучающая часть.
Сравниваем три числа на признак:

- **Шепли** с $v(S) = P_D\big(f(x_S, X_R) = \hat y\big)$. Точный, 16 коалиций.
  По Proposition 1 это достаточность.
- **Необходимость признака:** заменяем признак значениями из обучающей
  выборки, остальные оставляем как у цветка. Какая доля замен меняет
  предсказание.
- **Необходимость пары лепестковых признаков** — то же, меняем оба сразу.


In [3]:
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

iris = load_iris()
names = ["sepal length", "sepal width", "petal length", "petal width"]
X_train, X_test, y_train, y_test = train_test_split(iris.data, iris.target, test_size=0.3,
                                                    stratify=iris.target, random_state=0)
model = RandomForestClassifier(n_estimators=200, random_state=0).fit(X_train, y_train)
print(f"точность на тесте: {model.score(X_test, y_test):.3f}   ({len(X_test)} цветков)\n")

def value(x, S, y_hat):
    Z = X_train.copy()
    Z[:, list(S)] = x[list(S)]
    return (model.predict(Z) == y_hat).mean()

def necessity(x, idx, y_hat):
    Z = np.repeat(x[None], len(X_train), axis=0)
    Z[:, idx] = X_train[:, idx]
    return (model.predict(Z) != y_hat).mean()

def explain(x):
    y_hat = model.predict(x[None])[0]
    phi = exact_shapley(lambda S: value(x, S, y_hat), list(range(4)))
    return y_hat, pd.DataFrame({"Шепли": [phi[i] for i in range(4)],
                                "PN признака": [necessity(x, [i], y_hat) for i in range(4)]}, index=names)

for n in (2, 36):
    y_hat, table = explain(X_test[n])
    print(f"тестовый цветок #{n}, предсказание: {iris.target_names[y_hat]}, "
          f"PN пары лепестковых признаков = {necessity(X_test[n], [2, 3], y_hat):.3f}")
    print(table.round(3).to_string(), "\n")


точность на тесте: 0.978   (45 цветков)



тестовый цветок #2, предсказание: setosa, PN пары лепестковых признаков = 0.667
              Шепли  PN признака
sepal length  0.083          0.0
sepal width   0.051          0.0
petal length  0.203          0.0
petal width   0.330          0.0 



тестовый цветок #36, предсказание: versicolor, PN пары лепестковых признаков = 0.657
              Шепли  PN признака
sepal length  0.160          0.0
sepal width   0.047          0.0
petal length  0.198          0.0
petal width   0.263          0.0 



In [4]:
# то же по всему тесту: (цветок, признак) с заметным вкладом, но ненужные
rows = []
for x in X_test:
    y_hat, table = explain(x)
    for name, r in table.iterrows():
        rows.append({"признак": name, "Шепли": r["Шепли"], "PN": r["PN признака"]})
    rows.append({"признак": "пара лепестковых", "Шепли": np.nan, "PN": necessity(x, [2, 3], y_hat)})
pairs = pd.DataFrame(rows)

single = pairs[pairs["признак"] != "пара лепестковых"]
big = single[single["Шепли"] > 0.05]
print(f"признаков с Шепли > 0.05: {len(big)} из {len(single)}")
print(f"из них с PN = 0 (заменили признак — ответ не изменился ни разу): {(big['PN'] == 0).mean():.1%}\n")
print(pairs.groupby("признак", sort=False)["PN"].agg(["mean", lambda s: (s == 0).mean()])
      .set_axis(["средняя PN", "доля цветков с PN = 0"], axis=1).round(3).to_string())


признаков с Шепли > 0.05: 133 из 180
из них с PN = 0 (заменили признак — ответ не изменился ни разу): 63.2%

                  средняя PN  доля цветков с PN = 0
признак                                            
sepal length           0.022                  0.956
sepal width            0.018                  0.978
petal length           0.133                  0.622
petal width            0.279                  0.333
пара лепестковых       0.666                  0.000


Цветок #2 (setosa) — вся теорема об избыточности на отложенных данных.
Лепестковые признаки получают основной вклад, а необходимость каждого
**ноль**: длина и ширина лепестка дублируют друг друга, и лес узнаёт setosa
по любому из них. Необходимость пары — две трети. Чашелистики получают
положительный Шепли при нулевой необходимости.

Цветок #36 (versicolor): у всех четырёх признаков положительный вклад,
необходимого среди них нет ни одного.

По всему тесту это не исключение: у 63 % признаков с заметным вкладом
(Шепли > 0.05) необходимость нулевая, а у пары лепестковых признаков
нулевой необходимости нет ни на одном цветке, в среднем PN пары 0.67.

Если бы вопрос был «что можно выкинуть из признаков, не изменив ответ»,
Шепли ответил бы «ничего, всё важно». Правильный ответ — «любой один».


## Что уносим во второй акт

1. **PN и PS — разные величины.** В EchoLeak у обеих защит PS почти 1,
   а PN в нашем случае почти 0.
2. **SHAP-подобные методы меряют достаточность** (Watson, Prop. 1).
   Про необходимость они молчат по построению, не из-за реализации.
3. **При избыточности они систематически врут в сторону «всё важно»**
   (EchoLeak, ирисы).
4. **Необходимость в конкретном случае — контрфактуал.** Посчитать её
   можно, только зная модель мира. У агента модели нет: необходимость
   придётся мерить вмешательством (акт 1) или записывать связь по ходу (акт 2).

Дальше то же самое, но для агента: у него компоненты — инструменты и шаги,
каждая коалиция — отдельный прогон модели, а исход стохастичен.